In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis


In [2]:
INPUT_CSV = "/Users/kartikmisra/Documents/ECO723 Project/transformed_data.csv"  # <-- change path if needed
 
df = pd.read_csv(INPUT_CSV, index_col=0, parse_dates=True)
df = df.asfreq("MS")


In [3]:
df.head()

,CPI,CoreCPI,PCE,CorePCE,IP,UNEM,WORK,HS,INC,SPREAD
DATE,,,,,,,,,,
1984-01-01,NaN,NaN,NaN,NaN,-0.302046,0.237734,-657.839553,82.716363,0.6,2.47
1984-02-01,5.862249,3.513943,-7.912281,7.539770,-0.148055,0.070068,-401.514862,449.170716,-1.3,2.45
1984-03-01,3.503652,4.669213,13.948162,4.546608,0.033757,0.102385,-357.144488,-144.380675,0.8,2.50
1984-04-01,4.655680,5.811013,12.714115,5.634049,0.271621,0.034665,-221.654864,47.025252,0.7,2.68
1984-05-01,2.321084,4.628970,8.272539,2.782101,0.485067,-0.233122,-142.947623,-26.638411,0.6,3.34


# Descriptive statistics

In [4]:
def compute_descriptive_stats(data, fisher_kurtosis=True):
    rows = []
    for col in data.columns:
        s = data[col].dropna()
        rows.append({
            "Series": col,
            "Mean": s.mean(),
            "Median": s.median(),
            "Maximum": s.max(),
            "Minimum": s.min(),
            "Std. Dev.": s.std(),
            "Skewness": skew(s),
            "Kurtosis": kurtosis(s, fisher=fisher_kurtosis) + (0 if fisher_kurtosis else 0),
            "Obs.": s.shape[0],
        })
    result = pd.DataFrame(rows).set_index("Series")
    return result.round(2)
 
 
desc_stats = compute_descriptive_stats(df, fisher_kurtosis=False)
print("\n=== Descriptive Statistics (Table 2) ===")
print(desc_stats)



=== Descriptive Statistics (Table 2) ===
         Mean  Median  Maximum  Minimum  Std. Dev.  Skewness  Kurtosis  Obs.
Series                                                                      
CPI      2.71    2.70    16.41   -21.44       3.08     -1.49     15.20   371
CoreCPI  2.74    2.74     9.90    -3.27       2.43      0.19      2.78   371
PCE      5.21    4.99    34.75   -24.92       6.10     -0.13      8.28   371
CorePCE  2.31    2.12     8.42    -6.84       1.51      0.13      7.30   371
IP       0.00    0.05     4.63    -7.42       1.50     -1.06      9.78   372
UNEM     0.00   -0.00     1.34    -1.18       0.37      0.51      5.15   372
WORK     0.00  -57.93  2091.30 -2190.52     721.40      0.06      4.42   372
HS       0.00   -0.63   449.17  -341.21      97.79      0.32      5.02   372
INC      0.24    0.20     2.40    -2.60       0.48     -0.20      8.89   372
SPREAD   1.44    1.45     3.61    -0.60       0.87     -0.04      2.23   372


In [5]:
desc_stats.to_csv("descriptive_stats.csv")
print("\nSaved: descriptive_stats.csv")


Saved: descriptive_stats.csv


In [6]:
inflation_candidates = [c for c in ["CPI", "CoreCPI", "PCE", "CorePCE"] if c in desc_stats.index]
if inflation_candidates:
    print("\n--- Volatility ranking among inflation series (Std. Dev., high to low) ---")
    print(desc_stats.loc[inflation_candidates, "Std. Dev."].sort_values(ascending=False))



--- Volatility ranking among inflation series (Std. Dev., high to low) ---
Series
PCE        6.10
CPI        3.08
CoreCPI    2.43
CorePCE    1.51
Name: Std. Dev., dtype: float64


# Train Test Split

In [7]:
TRAIN_START = "1984-01-01"
TRAIN_END = "2013-12-31"
TEST_START = "2014-01-01"
TEST_END = "2014-12-31"
 
train = df.loc[TRAIN_START:TRAIN_END]
test = df.loc[TEST_START:TEST_END]
 
print(f"\nTrain set: {train.index.min()} to {train.index.max()} "
      f"({train.shape[0]} obs)")
print(f"Test set:  {test.index.min()} to {test.index.max()} "
      f"({test.shape[0]} obs)")



Train set: 1984-01-01 00:00:00 to 2013-12-01 00:00:00 (360 obs)
Test set:  2014-01-01 00:00:00 to 2014-12-01 00:00:00 (12 obs)


# AR

In [8]:
from statsmodels.tsa.ar_model import AutoReg

In [9]:
paper_lags = {"CPI": 2, "CoreCPI": 12, "PCE": 1, "CorePCE": 5}

In [10]:
def ar_forecast_12m(train_series, lag, n_periods=12):
    """
    Fit AR(lag) on train_series, return (fitted_model, forecast_series) where
    forecast_series has n_periods values immediately following the end of train_series.
    """
    model = AutoReg(train_series, lags=lag, old_names=False).fit()
    start = len(train_series)
    end = start + n_periods - 1
    fc = model.predict(start=start, end=end)
    return model, fc

In [11]:
def rmse_at_horizons(actual, forecast, horizons=(3, 6, 9, 12)):
    actual = actual.values
    forecast = forecast.values
    out = {}
    for h in horizons:
        err = actual[:h] - forecast[:h]
        out[h] = float(np.sqrt(np.mean(err ** 2)))
    return out

In [12]:
horizons = [3, 6, 9, 12]

ar_models = {}
ar_forecasts = {}
ar_rmse = {}

for col, lag in paper_lags.items():
    train_s = train[col].dropna()
    test_s = test[col].dropna()

    model, fc = ar_forecast_12m(train_s, lag=lag, n_periods=len(test_s))
    fc.index = test_s.index  # align forecast index with test dates

    ar_models[col] = model
    ar_forecasts[col] = fc
    ar_rmse[col] = rmse_at_horizons(test_s, fc, horizons)

print("AR lags used (fixed, from paper):")
for col, p in paper_lags.items():
    print(f"  {col}: p = {p}")

AR lags used (fixed, from paper):
  CPI: p = 2
  CoreCPI: p = 12
  PCE: p = 1
  CorePCE: p = 5


In [13]:
ar_rmse_df = pd.DataFrame(ar_rmse).T
ar_rmse_df.columns = [f"h={h}" for h in horizons]
ar_rmse_df.index.name = "Inflation"

print("\n=== AR model RMSE by horizon (paper's lags; compare with Table 3, 'AR' column) ===")
print(ar_rmse_df.round(2))



=== AR model RMSE by horizon (paper's lags; compare with Table 3, 'AR' column) ===
            h=3   h=6   h=9  h=12
Inflation                        
CPI        0.91  0.86  1.58  2.87
CoreCPI    0.82  1.16  1.13  1.13
PCE        4.64  3.33  3.25  3.25
CorePCE    0.83  0.65  0.75  0.84


# Naive Bayes

In [14]:
inflation_series = ["CPI", "CoreCPI", "PCE", "CorePCE"]

naive_forecasts = {}
naive_rmse = {}

for col in inflation_series:
    train_s = train[col].dropna()
    test_s = test[col].dropna()

    last_val = train_s.iloc[-1]
    fc = pd.Series([last_val] * len(test_s), index=test_s.index)

    naive_forecasts[col] = fc
    naive_rmse[col] = rmse_at_horizons(test_s, fc, horizons)

naive_rmse_df = pd.DataFrame(naive_rmse).T
naive_rmse_df.columns = [f"h={h}" for h in horizons]
naive_rmse_df.index.name = "Inflation"

print("=== Naive model RMSE by horizon (compare with Table 3, 'Naive' column) ===")
print(naive_rmse_df.round(2))

=== Naive model RMSE by horizon (compare with Table 3, 'Naive' column) ===
            h=3   h=6   h=9  h=12
Inflation                        
CPI        1.16  1.17  1.89  3.17
CoreCPI    4.55  4.06  3.68  3.42
PCE        4.51  3.40  3.32  3.19
CorePCE    0.50  0.57  0.61  0.55


# VAR

In [15]:
from statsmodels.tsa.api import VAR

In [16]:
VAR_PREDICTORS = ["UNEM", "IP", "WORK", "HS", "INC", "SPREAD"]
VAR_LAG = 1  # per paper: "selected as one by Schwarz criteria"

def var_forecast_12m(train_df, target_col, predictors, lag=1, n_periods=12):
    cols = [target_col] + predictors
    endog = train_df[cols].dropna()
    model = VAR(endog).fit(lag)

    last_obs = endog.values[-lag:]
    fc_array = model.forecast(y=last_obs, steps=n_periods)

    target_idx = cols.index(target_col)
    return model, pd.Series(fc_array[:, target_idx])

var_models, var_forecasts, var_rmse = {}, {}, {}
for col in inflation_series:
    test_s = test[col].dropna()
    model, fc = var_forecast_12m(train, col, VAR_PREDICTORS, lag=VAR_LAG, n_periods=len(test_s))
    fc.index = test_s.index
    var_models[col], var_forecasts[col] = model, fc
    var_rmse[col] = rmse_at_horizons(test_s, fc, horizons)

In [17]:
var_rmse_df = pd.DataFrame(var_rmse).T
var_rmse_df.columns = [f"h={h}" for h in horizons]
var_rmse_df.index.name = "Inflation"

print("=== VAR model RMSE by horizon (compare with Table 3, 'VAR' column) ===")
print(var_rmse_df.round(2))

=== VAR model RMSE by horizon (compare with Table 3, 'VAR' column) ===
            h=3   h=6   h=9  h=12
Inflation                        
CPI        0.81  0.75  1.49  2.81
CoreCPI    1.37  1.27  1.47  2.08
PCE        4.76  3.40  3.30  3.29
CorePCE    1.18  0.94  1.01  1.10


# ARDL

In [18]:
from statsmodels.tsa.ardl import ARDL, ardl_select_order

In [19]:
def ardl_forecast_12m(train_df, test_df, target_col, predictors, maxlag=6, maxorder=3, n_periods=12):
    cols = [target_col] + predictors
    d = train_df[cols].dropna()
    endog, exog = d[target_col], d[predictors]

    sel = ardl_select_order(endog, maxlag=maxlag, exog=exog, maxorder=maxorder, ic="bic", trend="c")
    ar_lags, dl_lags = sel.ar_lags, sel.dl_lags

    model = ARDL(endog, lags=ar_lags, exog=exog, order=dl_lags, trend="c", causal=False)
    results = model.fit()

    start = len(d); end = start + n_periods - 1
    fc = results.predict(start=start, end=end, exog_oos=test_df[predictors].iloc[:n_periods])
    return ar_lags, dl_lags, results, fc

In [20]:
ardl_specs = {}
ardl_models = {}
ardl_forecasts = {}
ardl_rmse = {}

for col in inflation_series:
    test_s = test[col].dropna()

    ar_lags, dl_lags, results, fc = ardl_forecast_12m(
        train, test, col, VAR_PREDICTORS,
        maxlag=6, maxorder=3, n_periods=len(test_s),
    )
    fc.index = test_s.index

    ardl_specs[col] = {"ar_lags": ar_lags, "dl_lags": dl_lags}
    ardl_models[col] = results
    ardl_forecasts[col] = fc
    ardl_rmse[col] = rmse_at_horizons(test_s, fc, horizons)

print("ARDL selected specs (BIC):")
for col, spec in ardl_specs.items():
    print(f"  {col}: own lags={spec['ar_lags']}, exog lags={spec['dl_lags']}")


/opt/miniconda3/envs/xgb_env/lib/python3.10/site-packages/statsmodels/tsa/ardl/model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: IP, HS, UNEM, INC, SPREAD.
  return _format_order(self.data.orig_exog, order, self._causal)
/opt/miniconda3/envs/xgb_env/lib/python3.10/site-packages/statsmodels/tsa/ardl/model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: IP, HS, UNEM, WORK, INC, SPREAD.
  return _format_order(self.data.orig_exog, order, self._causal)
/opt/miniconda3/envs/xgb_env/lib/python3.10/site-packages/statsmodels/tsa/ardl/model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: IP, HS, UNEM, WORK, SPREAD.
  return _format_order(self.data.orig_exog, order, self._causal)


ARDL selected specs (BIC):
  CPI: own lags=[1, 2], exog lags={'WORK': [0, 1]}
  CoreCPI: own lags=[1, 2, 3, 4, 5, 6], exog lags={}
  PCE: own lags=[1], exog lags={'INC': [0, 1]}
  CorePCE: own lags=[1, 2, 3, 4, 5, 6], exog lags={}


/opt/miniconda3/envs/xgb_env/lib/python3.10/site-packages/statsmodels/tsa/ardl/model.py:455: SpecificationWarning: exog contains variables that are missing from the order dictionary.  Missing keys: IP, HS, UNEM, WORK, INC, SPREAD.
  return _format_order(self.data.orig_exog, order, self._causal)


In [21]:
ardl_rmse_df = pd.DataFrame(ardl_rmse).T
ardl_rmse_df.columns = [f"h={h}" for h in horizons]
ardl_rmse_df.index.name = "Inflation"

print("\n=== ARDL model RMSE by horizon (compare with Table 3, 'ARDL' column) ===")
print(ardl_rmse_df.round(2))



=== ARDL model RMSE by horizon (compare with Table 3, 'ARDL' column) ===
            h=3   h=6   h=9  h=12
Inflation                        
CPI        0.69  0.95  1.66  3.05
CoreCPI    0.96  1.04  1.20  1.58
PCE        2.93  2.22  2.13  2.61
CorePCE    0.74  0.56  0.67  0.75


# kNN

In [22]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

In [23]:
KNN_K = 3  # paper reports k=3 as best after trying several values

def build_knn_features(df, target_col, predictors):
    feat = pd.DataFrame(index=df.index)
    feat["target_lag1"] = df[target_col].shift(1)
    for p in predictors:
        feat[p] = df[p]
    feat["y"] = df[target_col]
    return feat.dropna()


In [24]:
def knn_forecast_12m(train_df, test_df, target_col, predictors, k=3, n_periods=12):
    feat_cols = ["target_lag1"] + predictors
    train_feat = build_knn_features(train_df, target_col, predictors)
    X_train, y_train = train_feat[feat_cols].values, train_feat["y"].values

    scaler = StandardScaler().fit(X_train)
    model = KNeighborsRegressor(n_neighbors=k).fit(scaler.transform(X_train), y_train)

    last_target = train_df[target_col].dropna().iloc[-1]
    test_predictors = test_df[predictors].iloc[:n_periods]

    preds, prev_target = [], last_target
    for i in range(n_periods):
        row = [prev_target] + list(test_predictors.iloc[i].values)
        pred = model.predict(scaler.transform([row]))[0]
        preds.append(pred)
        prev_target = pred  # recursive: feed forecast back as next lag

    return model, scaler, pd.Series(preds, index=test_predictors.index)

In [25]:
knn_models = {}
knn_forecasts = {}
knn_rmse = {}

for col in inflation_series:
    test_s = test[col].dropna()

    model, scaler, fc = knn_forecast_12m(
        train, test, col, VAR_PREDICTORS, k=KNN_K, n_periods=len(test_s)
    )

    knn_models[col] = model
    knn_forecasts[col] = fc
    knn_rmse[col] = rmse_at_horizons(test_s, fc, horizons)

knn_rmse_df = pd.DataFrame(knn_rmse).T
knn_rmse_df.columns = [f"h={h}" for h in horizons]
knn_rmse_df.index.name = "Inflation"

print("=== k-NN model RMSE by horizon (k=3; compare with Table 3, 'k-NN' column) ===")
print(knn_rmse_df.round(2))

=== k-NN model RMSE by horizon (k=3; compare with Table 3, 'k-NN' column) ===
            h=3   h=6   h=9  h=12
Inflation                        
CPI        0.96  0.77  1.97  2.97
CoreCPI    1.25  1.55  1.64  1.95
PCE        2.36  1.92  1.82  2.22
CorePCE    1.63  1.21  1.05  0.93


# SVR

In [26]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit


In [27]:
def svr_forecast_12m(train_df, test_df, target_col, predictors, n_periods=12):
    feat_cols = ["target_lag1"] + predictors

    train_feat = build_knn_features(train_df, target_col, predictors)  # reuse from k-NN section
    X_train = train_feat[feat_cols].values
    y_train = train_feat["y"].values

    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)

    param_grid = {
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", 0.01, 0.1, 1],
        "epsilon": [0.01, 0.1, 0.5],
    }
    tscv = TimeSeriesSplit(n_splits=5)
    grid = GridSearchCV(
        SVR(kernel="rbf"), param_grid, cv=tscv,
        scoring="neg_root_mean_squared_error", n_jobs=-1,
    )
    grid.fit(X_train_s, y_train)
    model = grid.best_estimator_

    last_target = train_df[target_col].dropna().iloc[-1]
    test_predictors = test_df[predictors].iloc[:n_periods]

    preds = []
    prev_target = last_target
    for i in range(n_periods):
        row = [prev_target] + list(test_predictors.iloc[i].values)
        row_s = scaler.transform([row])
        pred = model.predict(row_s)[0]
        preds.append(pred)
        prev_target = pred

    fc = pd.Series(preds, index=test_predictors.index)
    return model, scaler, grid.best_params_, fc


In [28]:
svr_models = {}
svr_forecasts = {}
svr_rmse = {}
svr_best_params = {}

for col in inflation_series:
    test_s = test[col].dropna()

    model, scaler, best_params, fc = svr_forecast_12m(
        train, test, col, VAR_PREDICTORS, n_periods=len(test_s)
    )

    svr_models[col] = model
    svr_forecasts[col] = fc
    svr_best_params[col] = best_params
    svr_rmse[col] = rmse_at_horizons(test_s, fc, horizons)

print("SVR best hyperparameters (grid search, TimeSeriesSplit CV on train):")
for col, p in svr_best_params.items():
    print(f"  {col}: {p}")


SVR best hyperparameters (grid search, TimeSeriesSplit CV on train):
  CPI: {'C': 100, 'epsilon': 0.01, 'gamma': 0.01}
  CoreCPI: {'C': 10, 'epsilon': 0.5, 'gamma': 0.01}
  PCE: {'C': 100, 'epsilon': 0.5, 'gamma': 0.01}
  CorePCE: {'C': 1, 'epsilon': 0.1, 'gamma': 0.1}


In [29]:
svr_rmse_df = pd.DataFrame(svr_rmse).T
svr_rmse_df.columns = [f"h={h}" for h in horizons]
svr_rmse_df.index.name = "Inflation"

print("\n=== SVR model RMSE by horizon (RBF kernel; compare with Table 3, 'SVR' column) ===")
print(svr_rmse_df.round(2))



=== SVR model RMSE by horizon (RBF kernel; compare with Table 3, 'SVR' column) ===
            h=3   h=6   h=9  h=12
Inflation                        
CPI        0.96  1.02  1.66  2.92
CoreCPI    1.46  1.35  1.46  2.03
PCE        2.96  2.32  2.22  2.67
CorePCE    0.80  0.67  0.79  0.89


# ANN

In [30]:
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)


In [31]:
def ann_forecast_12m(train_df, test_df, target_col, predictors, n_periods=12, random_state=42):
    """
    ANN forecast using the same feature setup as k-NN/SVR (target_lag1 + VAR_PREDICTORS).
    Paper specifies: 1 hidden layer, 3 neurons, learning rate 0.3, momentum 0.2, best activation
    function chosen by trial. We grid-search activation + a couple of learning-rate/momentum
    combos around the paper's reported values, scoring on TimeSeriesSplit CV (RMSE), then refit
    the best config on the full training set and forecast recursively.
    """
    feat_cols = ["target_lag1"] + predictors

    train_feat = build_knn_features(train_df, target_col, predictors)  # reuse from k-NN section
    X_train = train_feat[feat_cols].values
    y_train = train_feat["y"].values

    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)

    param_grid = {
        "activation": ["identity", "logistic", "tanh", "relu"],
        "learning_rate_init": [0.3, 0.1, 0.01],
        "momentum": [0.2, 0.9],
    }
    tscv = TimeSeriesSplit(n_splits=5)

    base_mlp = MLPRegressor(
        hidden_layer_sizes=(3,),       # paper: one hidden layer, 3 neurons
        solver="sgd",                  # closest analogue to paper's LR + momentum training
        max_iter=2000,
        random_state=random_state,
    )
    grid = GridSearchCV(
        base_mlp, param_grid, cv=tscv,
        scoring="neg_root_mean_squared_error", n_jobs=-1,
    )
    grid.fit(X_train_s, y_train)
    model = grid.best_estimator_

    last_target = train_df[target_col].dropna().iloc[-1]
    test_predictors = test_df[predictors].iloc[:n_periods]

    preds = []
    prev_target = last_target
    for i in range(n_periods):
        row = [prev_target] + list(test_predictors.iloc[i].values)
        row_s = scaler.transform([row])
        pred = model.predict(row_s)[0]
        preds.append(pred)
        prev_target = pred  # recursive: feed forecast back as next lag

    fc = pd.Series(preds, index=test_predictors.index)
    return model, scaler, grid.best_params_, fc


In [32]:
ann_models = {}
ann_forecasts = {}
ann_rmse = {}
ann_best_params = {}

for col in inflation_series:
    test_s = test[col].dropna()

    model, scaler, best_params, fc = ann_forecast_12m(
        train, test, col, VAR_PREDICTORS, n_periods=len(test_s)
    )

    ann_models[col] = model
    ann_forecasts[col] = fc
    ann_best_params[col] = best_params
    ann_rmse[col] = rmse_at_horizons(test_s, fc, horizons)

print("ANN best hyperparameters (grid search, TimeSeriesSplit CV on train):")
for col, p in ann_best_params.items():
    print(f"  {col}: {p}")

/opt/miniconda3/envs/xgb_env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/miniconda3/envs/xgb_env/lib/python3.10/site-packages/sklearn/neural_network/_base.py:188: RuntimeWarning: overflow encountered in square
  0.5 * np.average((y_true - y_pred) ** 2, weights=sample_weight, axis=0).mean()
/opt/miniconda3/envs/xgb_env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/miniconda3/envs/xgb_env/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/opt/miniconda3/envs/xgb_env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/miniconda3/envs/xgb_env/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encoun

ANN best hyperparameters (grid search, TimeSeriesSplit CV on train):
  CPI: {'activation': 'identity', 'learning_rate_init': 0.01, 'momentum': 0.9}
  CoreCPI: {'activation': 'relu', 'learning_rate_init': 0.1, 'momentum': 0.2}
  PCE: {'activation': 'identity', 'learning_rate_init': 0.01, 'momentum': 0.2}
  CorePCE: {'activation': 'relu', 'learning_rate_init': 0.01, 'momentum': 0.2}


In [33]:
ann_rmse_df = pd.DataFrame(ann_rmse).T
ann_rmse_df.columns = [f"h={h}" for h in horizons]
ann_rmse_df.index.name = "Inflation"

print("\n=== ANN model RMSE by horizon (compare with Table 3, 'ANN' column) ===")
print(ann_rmse_df.round(2))



=== ANN model RMSE by horizon (compare with Table 3, 'ANN' column) ===
            h=3   h=6   h=9  h=12
Inflation                        
CPI        0.79  0.87  1.56  2.83
CoreCPI    1.60  1.45  1.48  1.95
PCE        2.76  2.14  2.09  2.60
CorePCE    0.82  0.79  0.87  0.94


# R^2

In [34]:
def r2_paper(actual, forecast, n_periods=12):
    """
    Replicates the paper's R^2 (Eq. 7-8): regress actual (Y) on forecast (Yhat)
    with NO intercept:  Y_i = a1 * Yhat_i + e_i
    R^2 here is the resulting goodness-of-fit measure (NOT the standard
    1 - SS_res/SS_tot you'd get from sklearn.metrics.r2_score, which compares
    against the mean of Y and includes an intercept).
    """
    y = np.asarray(actual)[:n_periods]
    yhat = np.asarray(forecast)[:n_periods]

    # a1_hat = sum(y*yhat) / sum(yhat^2)  -> OLS slope, no intercept
    a1_hat = np.sum(y * yhat) / np.sum(yhat ** 2)
    resid = y - a1_hat * yhat

    ss_res = np.sum(resid ** 2)
    ss_tot = np.sum(y ** 2)  # paper's denominator uses sum(Y^2), consistent with no-intercept regression
    r2 = 1 - ss_res / ss_tot
    return r2


def r2_sklearn_style(actual, forecast, n_periods=12):
    """
    Standard R^2 = 1 - SS_res/SS_tot, where SS_tot is computed against the MEAN
    of actual (the usual definition, e.g. sklearn.metrics.r2_score).
    Included for reference/comparison -- the paper's Table 4 uses r2_paper above.
    """
    y = np.asarray(actual)[:n_periods]
    yhat = np.asarray(forecast)[:n_periods]

    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot


In [35]:
# Map of model name -> forecasts dict (each dict: {inflation_col: forecast_series})
all_forecasts = {
    "AR": ar_forecasts,
    "Naive": naive_forecasts,
    "VAR": var_forecasts,
    "ARDL": ardl_forecasts,
    "k-NN": knn_forecasts,
    "SVR": svr_forecasts,
    "ANN": ann_forecasts,
}

r2_results = {}      # paper-style R^2 (Eq. 7-8, no-intercept regression)
r2_results_sklearn = {}  # standard R^2, for reference

for model_name, fc_dict in all_forecasts.items():
    row = {}
    row_sklearn = {}
    for col in inflation_series:
        test_s = test[col].dropna()
        fc = fc_dict[col]

        # align on common dates just in case, then take first 12 (h=12 horizon)
        common_idx = test_s.index.intersection(fc.index)
        actual_12 = test_s.loc[common_idx].iloc[:12]
        forecast_12 = fc.loc[common_idx].iloc[:12]

        row[col] = r2_paper(actual_12, forecast_12, n_periods=12)
        row_sklearn[col] = r2_sklearn_style(actual_12, forecast_12, n_periods=12)

    r2_results[model_name] = row
    r2_results_sklearn[model_name] = row_sklearn


In [36]:
r2_df = pd.DataFrame(r2_results).T          # rows = models, cols = inflation series (matches Table 4 layout)
r2_df = r2_df[inflation_series]              # enforce CPI, CoreCPI, PCE, CorePCE column order
r2_df.index.name = "Model"

print("=== R^2 (paper's Eq. 7-8 definition), Horizon = 12 (compare with Table 4) ===")
print(r2_df.round(2))

=== R^2 (paper's Eq. 7-8 definition), Horizon = 12 (compare with Table 4) ===
        CPI  CoreCPI   PCE  CorePCE
Model                              
AR     0.11     0.79  0.65     0.85
Naive  0.10     0.48  0.67     0.86
VAR    0.10     0.43  0.64     0.86
ARDL   0.09     0.59  0.85     0.85
k-NN   0.06     0.42  0.82     0.67
SVR    0.13     0.43  0.85     0.85
ANN    0.13     0.42  0.85     0.86
